# 估值类合成

## 多类合成法基础指标结果汇总

In [2]:
"""
华泰金工《因子合成方法实证分析》——估值类因子合成复现（BigQuant 3.0）

复现对象
--------
原始估值因子：EP、EPcut、BP、SP
九类复合因子：
1. fac_eqwt          等权
2. fac_ret           历史因子收益率加权
3. fac_ret_half      历史因子收益率半衰加权
4. fac_ic            历史 RankIC 加权
5. fac_ic_half       历史 RankIC 半衰加权
6. fac_maxicir_samp  最大化 IC_IR（样本协方差）
7. fac_maxicir1      最大化 IC_IR（Ledoit-Wolf 压缩协方差）
8. fac_maxic         最大化 IC
9. fac_pca1          第一主成分

时间对齐
--------
每月最后一个交易日形成因子，以该月末至下一月末的后复权收益作为标签。
动态权重只使用截至当前月末已经完整实现的过去 12 个月信息，不使用未来数据。

输出
----
直接展示与研报图表 4 同结构的表格；所有结果同时保留在内存对象中，不自动保存文件。

重要说明
--------
1. 研报未披露半衰期 H 的最终取值，只说明可取 1、2、4 等；本代码默认 H=4 个月，
   可通过 HALF_LIFE_MONTHS 修改。这是与原报告无法做到逐点完全复现的参数之一。
2. BigQuant 与研报 Wind 数据源、财务口径、行业历史映射和复权实现可能不同，数值不会
   保证与论文表格完全一致，但方法、样本频率、评价框架与防未来函数时序保持一致。
3. 本代码是因子研究诊断，不是交易回测，因此不涉及交易成本和成交约束。
"""

import time
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA

from bigquant import dai


# =============================================================================
# 1. 集中参数
# =============================================================================

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"

# IC 均值、IC 协方差矩阵和动态加权统一使用过去 12 个完整月度截面。
LOOKBACK_MONTHS = 12

# 研报未明确给出 H；4 个月是文中列举的可选值之一。
HALF_LIFE_MONTHS = 4

MAD_MULTIPLIER = 5.0
MIN_CROSS_SECTION = 80
PROGRESS_EVERY = 8

RAW_FACTORS = ["EP", "EPcut", "BP", "SP"]
COMPOSITE_FACTORS = [
    "fac_eqwt",
    "fac_ret",
    "fac_ret_half",
    "fac_ic",
    "fac_ic_half",
    "fac_maxicir_samp",
    "fac_maxicir1",
    "fac_maxic",
    "fac_pca1",
]
ALL_FACTORS = RAW_FACTORS + COMPOSITE_FACTORS

FACTOR_MEANINGS = {
    "EP": "净利润（TTM）/总市值",
    "EPcut": "扣非后净利润（TTM）/总市值",
    "BP": "净资产/总市值",
    "SP": "营业收入（TTM）/总市值",
    "fac_eqwt": "等权复合因子",
    "fac_ret": "历史收益率加权复合因子",
    "fac_ret_half": "历史收益率半衰加权复合因子",
    "fac_ic": "历史 IC 加权复合因子",
    "fac_ic_half": "历史 IC 半衰加权复合因子",
    "fac_maxicir_samp": "最大化 IC_IR 复合因子（样本）",
    "fac_maxicir1": "最大化 IC_IR 复合因子（压缩）",
    "fac_maxic": "最大化 IC 复合因子",
    "fac_pca1": "第一主成分复合因子",
}


@dataclass
class Progress:
    stage: str
    total: int
    every: int = PROGRESS_EVERY

    def __post_init__(self) -> None:
        self.start_time = time.time()

    def update(self, completed: int, current: object = "") -> None:
        if completed != 1 and completed != self.total and completed % self.every != 0:
            return
        elapsed = time.time() - self.start_time
        speed = completed / elapsed if elapsed > 0 else np.nan
        eta = (self.total - completed) / speed if speed and np.isfinite(speed) else np.nan
        eta_text = f"{eta:.1f}s" if np.isfinite(eta) else "--"
        print(
            f"[{self.stage}] {completed}/{self.total} ({completed / self.total:.1%}) "
            f"当前={current} 已用={elapsed:.1f}s 预计剩余={eta_text}"
        )


# =============================================================================
# 2. BigQuant 数据读取与月度标签
# =============================================================================

def load_monthly_data() -> pd.DataFrame:
    """读取全 A 日频数据后，在 SQL 端保留各股票每月最后一个交易日。"""

    # 多取当月月初，确保 START_DATE 所在月能正确识别月末；不向前读取研究样本。
    query_start = pd.Timestamp(START_DATE).replace(day=1).strftime("%Y-%m-%d")

    sql = """
    WITH daily AS (
        SELECT
            date,
            instrument,
            close,
            total_market_cap,
            float_market_cap,
            net_profit_ttm,
            net_profit_deducted_ttm,
            total_owner_equity_lf,
            operating_revenue_ttm,
            sw2014_level1 AS industry,
            st_status,
            suspended,
            LEAD(suspended, 1) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS next_day_suspended
        FROM cn_stock_prefactors
        WHERE date >= $query_start AND date <= $end_date
    ), ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY instrument, YEAR(date), MONTH(date)
                ORDER BY date DESC
            ) AS month_end_rank
        FROM daily
    )
    SELECT
        date, instrument, close,
        total_market_cap, float_market_cap,
        net_profit_ttm, net_profit_deducted_ttm,
        total_owner_equity_lf, operating_revenue_ttm,
        industry, st_status, suspended, next_day_suspended
    FROM ranked
    WHERE month_end_rank = 1
    ORDER BY date, instrument
    """

    print("[数据] 正在读取 BigQuant 月末截面……")
    df = dai.query(
        sql,
        filters={"date": [query_start, END_DATE]},
        params={"query_start": query_start, "end_date": END_DATE},
    ).df()

    if df.empty:
        raise RuntimeError("BigQuant 查询结果为空，请检查数据权限、日期和表字段。")

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    # 月末到下一月末收益。若下一条记录不是紧邻的自然月，则不把跨月缺口当作月收益。
    by_stock = df.groupby("instrument", sort=False)
    df["return_end_date"] = by_stock["date"].shift(-1)
    df["next_close"] = by_stock["close"].shift(-1)
    df["forward_return"] = df["next_close"] / df["close"] - 1.0

    current_period = df["date"].dt.to_period("M")
    expected_next = current_period + 1
    actual_next = df["return_end_date"].dt.to_period("M")
    invalid_gap = actual_next.notna() & (actual_next != expected_next)
    df.loc[invalid_gap, ["forward_return", "return_end_date"]] = np.nan

    # 研报：全 A；剔除 ST/PT；剔除截面期下一交易日停牌股票。
    universe = (
        (df["date"] >= pd.Timestamp(START_DATE))
        & (df["date"] <= pd.Timestamp(END_DATE))
        & (df["st_status"] == 0)
        & (df["next_day_suspended"] == 0)
        & (df["total_market_cap"] > 0)
        & (df["float_market_cap"] > 0)
        & df["industry"].notna()
    )
    df = df.loc[universe].copy()

    # 四个原始估值因子。
    df["EP"] = df["net_profit_ttm"] / df["total_market_cap"]
    df["EPcut"] = df["net_profit_deducted_ttm"] / df["total_market_cap"]
    df["BP"] = df["total_owner_equity_lf"] / df["total_market_cap"]
    df["SP"] = df["operating_revenue_ttm"] / df["total_market_cap"]

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    print(
        f"[数据] 完成：{len(df):,} 行，{df['date'].nunique()} 个月末，"
        f"{df['date'].min().date()} 至 {df['date'].max().date()}"
    )
    return df


# =============================================================================
# 3. 截面处理、回归法与 RankIC
# =============================================================================

def mad_winsorize(s: pd.Series, n: float = MAD_MULTIPLIER) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").astype(float)
    valid = x.dropna()
    if valid.empty:
        return x
    median = valid.median()
    mad = (valid - median).abs().median()
    if not np.isfinite(mad) or mad <= 0:
        return x
    return x.clip(median - n * mad, median + n * mad)


def zscore(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").astype(float)
    std = x.std(ddof=1)
    if not np.isfinite(std) or std <= 0:
        return pd.Series(np.nan, index=s.index, dtype=float)
    return (x - x.mean()) / std


def prepare_raw_exposures(df: pd.DataFrame) -> pd.DataFrame:
    """逐月对原始因子做 5MAD 去极值和 Z-score；保留原始缺失值。"""
    pieces: List[pd.DataFrame] = []
    dates = sorted(df["date"].unique())
    progress = Progress("原始因子截面处理", len(dates))

    for i, date in enumerate(dates, 1):
        g = df.loc[df["date"] == date].copy()
        for factor in RAW_FACTORS:
            g[factor] = zscore(mad_winsorize(g[factor]))
        pieces.append(g)
        progress.update(i, pd.Timestamp(date).date())

    return pd.concat(pieces, ignore_index=True)


def _add_constant(X: pd.DataFrame) -> pd.DataFrame:
    out = X.copy()
    out.insert(0, "const", 1.0)
    return out.astype(float)


def _design_matrix(g: pd.DataFrame, factor: str) -> Tuple[pd.DataFrame, pd.Series]:
    industry = pd.get_dummies(
        g["industry"].astype(str), prefix="ind", drop_first=True, dtype=float
    )
    controls = pd.DataFrame(
        {
            factor: g[factor].astype(float),
            "ln_total_market_cap": np.log(g["total_market_cap"].astype(float)),
        },
        index=g.index,
    )
    X = pd.concat([controls, industry], axis=1).astype(float)
    X = _add_constant(X)
    y = g["forward_return"].astype(float)
    return X, y


def _weighted_regression(
    y: pd.Series,
    X: pd.DataFrame,
    weights: pd.Series,
) -> Tuple[pd.Series, pd.Series]:
    """用 NumPy 完成 WLS，并返回系数与经典 WLS t 值。"""
    Xv = X.to_numpy(dtype=float)
    yv = y.to_numpy(dtype=float)
    wv = weights.to_numpy(dtype=float)
    sqrt_w = np.sqrt(wv)
    Xw = Xv * sqrt_w[:, None]
    yw = yv * sqrt_w

    beta, _, rank, _ = np.linalg.lstsq(Xw, yw, rcond=None)
    resid_w = yw - Xw @ beta
    dof = len(yv) - int(rank)
    if dof <= 0:
        raise np.linalg.LinAlgError("WLS 自由度不足")
    sigma2 = float(resid_w @ resid_w) / dof
    covariance = sigma2 * np.linalg.pinv(Xw.T @ Xw)
    standard_error = np.sqrt(np.maximum(np.diag(covariance), 0.0))
    t_value = np.divide(
        beta,
        standard_error,
        out=np.full_like(beta, np.nan, dtype=float),
        where=standard_error > 0,
    )
    return pd.Series(beta, index=X.columns), pd.Series(t_value, index=X.columns)


def cross_section_metrics(g: pd.DataFrame, factor: str) -> Dict[str, float]:
    """计算单个月份的 WLS 因子收益、t 值和行业市值中性 RankIC。"""
    cols = [
        factor,
        "forward_return",
        "total_market_cap",
        "float_market_cap",
        "industry",
    ]
    s = g[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(s) < MIN_CROSS_SECTION or s[factor].nunique() < 3:
        return {"factor_return": np.nan, "t_value": np.nan, "rank_ic": np.nan, "n": len(s)}

    # 回归法：流通市值平方根作为 WLS 权重，加入行业和对数总市值控制变量。
    try:
        X, y = _design_matrix(s, factor)
        weights = np.sqrt(s["float_market_cap"].astype(float))
        params, tvalues = _weighted_regression(y, X, weights)
        factor_return = float(params[factor])
        t_value = float(tvalues[factor])
    except Exception:
        factor_return, t_value = np.nan, np.nan

    # IC 法：先对行业和市值做 OLS 中性化，再与下期收益计算 Spearman RankIC。
    try:
        industry = pd.get_dummies(
            s["industry"].astype(str), prefix="ind", drop_first=True, dtype=float
        )
        neutral_X = pd.concat(
            [
                pd.Series(
                    np.log(s["total_market_cap"].astype(float)),
                    index=s.index,
                    name="ln_total_market_cap",
                ),
                industry,
            ],
            axis=1,
        ).astype(float)
        neutral_X = _add_constant(neutral_X)
        neutral_beta = np.linalg.lstsq(
            neutral_X.to_numpy(dtype=float),
            s[factor].to_numpy(dtype=float),
            rcond=None,
        )[0]
        residual = pd.Series(
            s[factor].to_numpy(dtype=float)
            - neutral_X.to_numpy(dtype=float) @ neutral_beta,
            index=s.index,
        )
        rank_ic = float(residual.corr(s["forward_return"], method="spearman"))
    except Exception:
        rank_ic = np.nan

    return {
        "factor_return": factor_return,
        "t_value": t_value,
        "rank_ic": rank_ic,
        "n": len(s),
    }


def evaluate_factors(
    panel: pd.DataFrame,
    factors: Iterable[str],
    dates: Optional[Iterable[pd.Timestamp]] = None,
    stage: str = "因子评价",
) -> pd.DataFrame:
    factor_list = list(factors)
    if dates is None:
        date_list = sorted(panel.loc[panel["forward_return"].notna(), "date"].unique())
    else:
        date_list = sorted(pd.to_datetime(list(dates)))

    total = len(date_list) * len(factor_list)
    progress = Progress(stage, total, every=max(PROGRESS_EVERY, len(factor_list)))
    records: List[Dict[str, object]] = []
    completed = 0

    for date in date_list:
        g = panel.loc[panel["date"] == date]
        for factor in factor_list:
            m = cross_section_metrics(g, factor)
            records.append({"date": pd.Timestamp(date), "factor": factor, **m})
            completed += 1
        progress.update(completed, pd.Timestamp(date).date())

    return pd.DataFrame(records).sort_values(["date", "factor"]).reset_index(drop=True)


# =============================================================================
# 4. 九类复合因子
# =============================================================================

def normalize_signed(raw: np.ndarray) -> np.ndarray:
    """按研报的历史收益/IC加权思想归一化；分母异常时退化为等权。"""
    x = np.asarray(raw, dtype=float)
    denom = x.sum()
    if not np.all(np.isfinite(x)) or abs(denom) < 1e-12:
        return np.repeat(1.0 / len(x), len(x))
    return x / denom


def exponential_time_weights(length: int, half_life: float) -> np.ndarray:
    # 输入历史按“最远 -> 最近”排列，最近一期权重最大。
    distance = np.arange(length - 1, -1, -1, dtype=float)
    w = np.power(0.5, distance / half_life)
    return w / w.sum()


def solve_nonnegative_ratio(mu: np.ndarray, cov: np.ndarray) -> np.ndarray:
    """求 max w'mu/sqrt(w'cov w), s.t. w>=0, sum(w)=1。"""
    mu = np.asarray(mu, dtype=float)
    cov = np.asarray(cov, dtype=float)
    n = len(mu)
    cov = (cov + cov.T) / 2.0 + np.eye(n) * 1e-10

    def objective(w: np.ndarray) -> float:
        variance = float(w @ cov @ w)
        if variance <= 1e-16:
            return 1e6
        return -float(w @ mu) / np.sqrt(variance)

    result = minimize(
        objective,
        x0=np.repeat(1.0 / n, n),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints=[{"type": "eq", "fun": lambda w: float(w.sum() - 1.0)}],
        options={"maxiter": 500, "ftol": 1e-12, "disp": False},
    )
    if not result.success or not np.all(np.isfinite(result.x)):
        positive = np.maximum(mu, 0.0)
        return normalize_signed(positive) if positive.sum() > 0 else np.repeat(1.0 / n, n)
    w = np.maximum(result.x, 0.0)
    return w / w.sum()


def _composite_value(X: pd.DataFrame, weights: np.ndarray) -> pd.Series:
    value = pd.Series(X.to_numpy(dtype=float) @ weights, index=X.index)
    return zscore(value)


def build_composite_factors(
    panel: pd.DataFrame,
    raw_metric_ts: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = panel.copy()
    for factor in COMPOSITE_FACTORS:
        out[factor] = np.nan

    rank_ic_hist = raw_metric_ts.pivot(index="date", columns="factor", values="rank_ic")
    factor_ret_hist = raw_metric_ts.pivot(
        index="date", columns="factor", values="factor_return"
    )
    rank_ic_hist = rank_ic_hist.reindex(columns=RAW_FACTORS)
    factor_ret_hist = factor_ret_hist.reindex(columns=RAW_FACTORS)

    dates = sorted(out["date"].unique())
    progress = Progress("九类复合因子", len(dates))
    weight_records: List[Dict[str, object]] = []

    for i, date in enumerate(dates, 1):
        date = pd.Timestamp(date)
        hist_mask = rank_ic_hist.index < date
        joined = pd.concat(
            {
                "ic": rank_ic_hist.loc[hist_mask],
                "ret": factor_ret_hist.loc[hist_mask],
            },
            axis=1,
        ).dropna(how="any")
        joined = joined.tail(LOOKBACK_MONTHS)

        if len(joined) < LOOKBACK_MONTHS:
            progress.update(i, f"{date.date()}（预热）")
            continue

        hist_ic = joined["ic"][RAW_FACTORS]
        hist_ret = joined["ret"][RAW_FACTORS]
        mu_ic = hist_ic.mean(axis=0).to_numpy(dtype=float)
        mu_ret = hist_ret.mean(axis=0).to_numpy(dtype=float)
        decay = exponential_time_weights(LOOKBACK_MONTHS, HALF_LIFE_MONTHS)

        current_idx = out.index[out["date"] == date]
        # 研报：原始因子标准化后，将缺失值置 0 再合成。
        X = out.loc[current_idx, RAW_FACTORS].fillna(0.0).astype(float)
        if len(X) < MIN_CROSS_SECTION:
            progress.update(i, f"{date.date()}（样本不足）")
            continue

        sample_ic_cov = hist_ic.cov(ddof=1).to_numpy(dtype=float)
        shrink_ic_cov = LedoitWolf().fit(hist_ic.to_numpy(dtype=float)).covariance_

        # 最大化 IC 使用当期标准化因子值的压缩协方差阵。
        value_cov = LedoitWolf().fit(X.to_numpy(dtype=float)).covariance_

        weights: Dict[str, np.ndarray] = {
            "fac_eqwt": np.repeat(1.0 / len(RAW_FACTORS), len(RAW_FACTORS)),
            "fac_ret": normalize_signed(mu_ret),
            "fac_ret_half": normalize_signed(
                (hist_ret.to_numpy(dtype=float) * decay[:, None]).sum(axis=0)
            ),
            "fac_ic": normalize_signed(mu_ic),
            "fac_ic_half": normalize_signed(
                (hist_ic.to_numpy(dtype=float) * decay[:, None]).sum(axis=0)
            ),
            "fac_maxicir_samp": solve_nonnegative_ratio(mu_ic, sample_ic_cov),
            "fac_maxicir1": solve_nonnegative_ratio(mu_ic, shrink_ic_cov),
            "fac_maxic": solve_nonnegative_ratio(mu_ic, value_cov),
        }

        pca = PCA(n_components=1)
        pca.fit(X.to_numpy(dtype=float))
        pca_loading = pca.components_[0].astype(float)
        # PCA 符号不唯一；统一为与“高估值因子值”同向，不使用未来收益定向。
        if pca_loading.sum() < 0:
            pca_loading *= -1.0
        weights["fac_pca1"] = pca_loading

        for method, w in weights.items():
            out.loc[current_idx, method] = _composite_value(X, w).to_numpy()
            for raw_factor, weight in zip(RAW_FACTORS, w):
                weight_records.append(
                    {
                        "date": date,
                        "method": method,
                        "raw_factor": raw_factor,
                        "weight": float(weight),
                    }
                )

        progress.update(i, date.date())

    return out, pd.DataFrame(weight_records)


# =============================================================================
# 5. 汇总与展示
# =============================================================================

def make_summary(metric_ts: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for factor in ALL_FACTORS:
        g = metric_ts.loc[metric_ts["factor"] == factor]
        t = g["t_value"].dropna()
        ret = g["factor_return"].dropna()
        ic = g["rank_ic"].dropna()
        ic_std = ic.std(ddof=1)
        rows.append(
            {
                "因子名称": factor,
                "因子含义": FACTOR_MEANINGS[factor],
                "|t|均值": t.abs().mean(),
                "|t|>2 占比": (t.abs() > 2).mean(),
                "t 均值": t.mean(),
                "因子收益率均值": ret.mean(),
                "Rank IC均值": ic.mean(),
                "Rank IC标准差": ic_std,
                "IC_IR": ic.mean() / ic_std if np.isfinite(ic_std) and ic_std > 0 else np.nan,
                "IC>0 占比": (ic > 0).mean(),
                "有效截面数": int(ic.notna().sum()),
            }
        )
    return pd.DataFrame(rows).set_index("因子名称")


def select_best_composite(summary: pd.DataFrame) -> Optional[str]:
    """在九类复合因子中选出 Rank IC 均值最高者；原始因子不参与。"""
    composite_rank_ic = summary.loc[COMPOSITE_FACTORS, "Rank IC均值"].dropna()
    return composite_rank_ic.idxmax() if not composite_rank_ic.empty else None


def display_summary(summary: pd.DataFrame) -> None:
    display_cols = [
        "因子含义",
        "|t|均值",
        "|t|>2 占比",
        "t 均值",
        "因子收益率均值",
        "Rank IC均值",
        "Rank IC标准差",
        "IC_IR",
        "IC>0 占比",
    ]

    # 红色高亮表达“当前复现结果中的最佳合成方法”，而不是机械复制研报中的固定行。
    # 按用户约定，仅在九类复合因子中，以 Rank IC 均值最高作为最有效标准。
    best_composite = select_best_composite(summary)

    if best_composite is not None:
        print(
            f"[最优合成方法] {best_composite}（{FACTOR_MEANINGS[best_composite]}），"
            f"判定标准：九类复合因子中 Rank IC 均值最高="
            f"{summary.loc[best_composite, 'Rank IC均值']:.2%}。"
        )

    try:
        from IPython.display import display

        style = (
            summary[display_cols]
            .style.format(
                {
                    "|t|均值": "{:.2f}",
                    "|t|>2 占比": "{:.2%}",
                    "t 均值": "{:.2f}",
                    "因子收益率均值": "{:.2%}",
                    "Rank IC均值": "{:.2%}",
                    "Rank IC标准差": "{:.2%}",
                    "IC_IR": "{:.2f}",
                    "IC>0 占比": "{:.2%}",
                }
            )
            .set_properties(**{"text-align": "center"})
            .set_properties(subset=["因子含义"], **{"text-align": "left"})
            .apply(
                lambda row: ["background-color: #eca0a0"] * len(row)
                if best_composite is not None and row.name == best_composite
                else [""] * len(row),
                axis=1,
            )
            .set_caption("红色行：九类复合因子中 Rank IC 均值最高的方法")
        )
        display(style)
    except Exception:
        printable = summary[display_cols].copy()
        for col in ["|t|>2 占比", "因子收益率均值", "Rank IC均值", "Rank IC标准差", "IC>0 占比"]:
            printable[col] = printable[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "NaN")
        for col in ["|t|均值", "t 均值", "IC_IR"]:
            printable[col] = printable[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NaN")
        print(printable.to_string())


def main() -> Dict[str, pd.DataFrame]:
    started = time.time()

    monthly = load_monthly_data()
    panel = prepare_raw_exposures(monthly)

    # 先计算四个原始因子的历史月度因子收益率与 RankIC，供动态合成使用。
    raw_metric_ts = evaluate_factors(
        panel,
        RAW_FACTORS,
        stage="原始因子历史指标",
    )

    factor_panel, weight_history = build_composite_factors(panel, raw_metric_ts)

    # 所有 9 个复合因子均存在且已有下一期收益的共同样本。
    common_date_check = factor_panel.groupby("date")[COMPOSITE_FACTORS].apply(
        lambda x: bool(x.notna().to_numpy().all())
    )
    common_dates = common_date_check.index[common_date_check].tolist()
    common_dates = [
        d
        for d in common_dates
        if factor_panel.loc[
            (factor_panel["date"] == d) & factor_panel["forward_return"].notna()
        ].shape[0]
        >= MIN_CROSS_SECTION
    ]
    if not common_dates:
        raise RuntimeError("没有形成可评价的共同截面，请检查样本期和 LOOKBACK_MONTHS。")

    metric_timeseries = evaluate_factors(
        factor_panel,
        ALL_FACTORS,
        dates=common_dates,
        stage="13个因子共同样本评价",
    )
    summary = make_summary(metric_timeseries)

    print(
        "\n[完成] 评价周期：月频；因子日在月末；标签为月末至下一月末收益；"
        f"共同有效截面 {len(common_dates)} 个（{pd.Timestamp(min(common_dates)).date()} 至 "
        f"{pd.Timestamp(max(common_dates)).date()}），总耗时 {time.time() - started:.1f}s。"
    )
    display_summary(summary)

    # 不自动保存。返回完整内存对象，便于继续检查权重和逐期指标。
    return {
        "summary": summary,
        "metric_timeseries": metric_timeseries,
        "factor_panel": factor_panel,
        "weight_history": weight_history,
        "raw_metric_timeseries": raw_metric_ts,
    }


if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    results = main()
    summary = results["summary"]
    metric_timeseries = results["metric_timeseries"]
    factor_panel = results["factor_panel"]
    weight_history = results["weight_history"]


[数据] 正在读取 BigQuant 月末截面……
[数据] 完成：247,663 行，53 个月末，2022-01-28 至 2026-05-29
[原始因子截面处理] 1/53 (1.9%) 当前=2022-01-28 已用=0.0s 预计剩余=0.4s
[原始因子截面处理] 8/53 (15.1%) 当前=2022-08-31 已用=0.1s 预计剩余=0.3s
[原始因子截面处理] 16/53 (30.2%) 当前=2023-04-28 已用=0.1s 预计剩余=0.2s
[原始因子截面处理] 24/53 (45.3%) 当前=2023-12-29 已用=0.2s 预计剩余=0.2s
[原始因子截面处理] 32/53 (60.4%) 当前=2024-08-30 已用=0.2s 预计剩余=0.1s
[原始因子截面处理] 40/53 (75.5%) 当前=2025-04-30 已用=0.3s 预计剩余=0.1s
[原始因子截面处理] 48/53 (90.6%) 当前=2025-12-31 已用=0.3s 预计剩余=0.0s
[原始因子截面处理] 53/53 (100.0%) 当前=2026-05-29 已用=0.3s 预计剩余=0.0s
[原始因子历史指标] 8/212 (3.8%) 当前=2022-02-28 已用=1.6s 预计剩余=40.1s
[原始因子历史指标] 16/212 (7.5%) 当前=2022-04-29 已用=3.2s 预计剩余=38.8s
[原始因子历史指标] 24/212 (11.3%) 当前=2022-06-30 已用=4.8s 预计剩余=37.4s
[原始因子历史指标] 32/212 (15.1%) 当前=2022-08-31 已用=6.5s 预计剩余=36.4s
[原始因子历史指标] 40/212 (18.9%) 当前=2022-10-31 已用=8.0s 预计剩余=34.3s
[原始因子历史指标] 48/212 (22.6%) 当前=2022-12-30 已用=9.6s 预计剩余=32.7s
[原始因子历史指标] 56/212 (26.4%) 当前=2023-02-28 已用=11.2s 预计剩余=31.1s
[原始因子历史指标] 64/212 (30.2%) 当前=2023-04-28 已用=12.8s 预计剩余=29.5s


,因子含义,|t|均值,|t|>2 占比,t 均值,因子收益率均值,Rank IC均值,Rank IC标准差,IC_IR,IC>0 占比
因子名称,,,,,,,,,
EP,净利润（TTM）/总市值,5.24,68.29%,0.95,0.03%,3.89%,11.86%,0.33,68.29%
EPcut,扣非后净利润（TTM）/总市值,5.11,65.85%,1.18,0.07%,3.88%,11.05%,0.35,65.85%
BP,净资产/总市值,5.27,70.73%,1.87,0.17%,6.32%,8.82%,0.72,75.61%
SP,营业收入（TTM）/总市值,4.15,73.17%,0.80,0.02%,3.72%,7.02%,0.53,70.73%
fac_eqwt,等权复合因子,5.63,75.61%,1.66,0.10%,5.56%,10.41%,0.53,65.85%
fac_ret,历史收益率加权复合因子,5.40,70.73%,1.44,0.08%,4.23%,10.34%,0.41,63.41%
fac_ret_half,历史收益率半衰加权复合因子,5.65,73.17%,1.75,0.14%,5.08%,10.35%,0.49,63.41%
fac_ic,历史 IC 加权复合因子,6.01,78.05%,1.64,0.08%,5.74%,10.83%,0.53,68.29%
fac_ic_half,历史 IC 半衰加权复合因子,5.80,73.17%,1.47,0.06%,5.25%,10.31%,0.51,65.85%


从汇总结果来看，最大化IC复合因子可以取得较好的RankIC均值，后续在做市值分组回测时可以重点关注该因子的表现

## 市值分组回测

In [3]:
"""
BigQuant：华泰估值因子 + 市值分组回测（参数化完整版）

支持的研报因子
--------------
原始因子：EP、EPcut、BP、SP
合成因子：fac_eqwt、fac_ret、fac_ret_half、fac_ic、fac_ic_half、
          fac_maxicir_samp、fac_maxicir1、fac_maxic、fac_pca1

时序
----
1. t 日收盘后，用 t 日可得的原始因子形成暴露。
2. 动态合成权重只使用收益标签已在 t 日收盘前（含 t 日收盘）实现完毕的
   最近 LOOKBACK_MONTHS=12 个完整月度截面。
3. t+1 交易日开盘调仓，每 REBALANCE_DAYS 个市场交易日重复一次。

市值分组
--------
每个信号日把可投资股票按总市值等数量划分为 15 组，第 1 组最小、
第 15 组最大。在 MARKET_CAP_GROUPS 指定的每组内独立选择因子 Top/Bottom k%，
合并后按股票全局等权。

脚本不自动保存任何运行结果。performance 和 RESEARCH_ARTIFACTS 保留在内存中。
"""

import math
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA

from bigquant import bigtrader, dai


# =============================================================================
# 1. 用户参数：修改这些参数即可切换因子和回测范围
# =============================================================================

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"
CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"

# 可选值见 ALL_REPORT_FACTORS。所有研报估值因子均定义为“数值越大越好”。
FACTOR_TO_TEST = "fac_maxicir1"
FACTOR_DIRECTION = 1  # 1：组内取最高 k%；-1：组内取最低 k%

MARKET_CAP_GROUPS = [15, 14, 13, 12]
TOP_K_PCT = 0.10
REBALANCE_DAYS = 60

# 研报动态权重窗口 T=12 个月；半衰期未完全披露，沿用复现代码默认 4 个月。
LOOKBACK_MONTHS = 12
HALF_LIFE_MONTHS = 4.0

MAD_MULTIPLIER = 5.0
MIN_CROSS_SECTION = 80
MIN_LIST_DAYS = 1
MARKET_CAP_GROUP_COUNT = 15
PROGRESS_EVERY = 20

# 交易成本与容量假设。
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5
VOLUME_LIMIT = 0.025
MIN_WEIGHT_CHANGE = 0.0005
LOG_EACH_REBALANCE = False

DATA_SOURCE = "cn_stock_prefactors"


RAW_FACTORS = ["EP", "EPcut", "BP", "SP"]
COMPOSITE_FACTORS = [
    "fac_eqwt",
    "fac_ret",
    "fac_ret_half",
    "fac_ic",
    "fac_ic_half",
    "fac_maxicir_samp",
    "fac_maxicir1",
    "fac_maxic",
    "fac_pca1",
]
ALL_REPORT_FACTORS = RAW_FACTORS + COMPOSITE_FACTORS
HISTORY_WEIGHTED_FACTORS = [
    factor for factor in COMPOSITE_FACTORS if factor not in ("fac_eqwt", "fac_pca1")
]

FACTOR_MEANINGS = {
    "EP": "净利润（TTM）/总市值",
    "EPcut": "扣非净利润（TTM）/总市值",
    "BP": "净资产/总市值",
    "SP": "营业收入（TTM）/总市值",
    "fac_eqwt": "等权复合因子",
    "fac_ret": "历史因子收益率加权复合因子",
    "fac_ret_half": "历史因子收益率半衰加权复合因子",
    "fac_ic": "历史 RankIC 加权复合因子",
    "fac_ic_half": "历史 RankIC 半衰加权复合因子",
    "fac_maxicir_samp": "最大化 IC_IR 复合因子（样本协方差）",
    "fac_maxicir1": "最大化 IC_IR 复合因子（压缩协方差）",
    "fac_maxic": "最大化 IC 复合因子",
    "fac_pca1": "第一主成分复合因子",
}


@dataclass(frozen=True)
class StrategyConfig:
    factor_name: str
    factor_direction: int
    market_cap_groups: tuple
    top_k_pct: float
    rebalance_days: int
    lookback_months: int
    half_life_months: float


# run_backtest 传入的配置会在 BigTrader initialize 中读取。
_ACTIVE_CONFIG = None

# 便于回测结束后检查权重和选股，不自动写文件。
RESEARCH_ARTIFACTS = {}


def make_config(
    factor_name,
    factor_direction,
    market_cap_groups,
    top_k_pct,
    rebalance_days,
    lookback_months,
    half_life_months,
):
    config = StrategyConfig(
        factor_name=str(factor_name),
        factor_direction=int(factor_direction),
        market_cap_groups=tuple(int(x) for x in market_cap_groups),
        top_k_pct=float(top_k_pct),
        rebalance_days=int(rebalance_days),
        lookback_months=int(lookback_months),
        half_life_months=float(half_life_months),
    )
    validate_config(config)
    return config


def validate_config(config):
    if config.factor_name not in ALL_REPORT_FACTORS:
        raise ValueError(
            f"未知因子 {config.factor_name!r}；可选值：{ALL_REPORT_FACTORS}"
        )
    if config.factor_direction not in (1, -1):
        raise ValueError("factor_direction 只能为 1 或 -1")
    if not config.market_cap_groups:
        raise ValueError("market_cap_groups 不能为空")
    invalid = sorted(set(config.market_cap_groups) - set(range(1, 16)))
    if invalid:
        raise ValueError(f"市值组必须位于 1~15，错误值：{invalid}")
    if len(set(config.market_cap_groups)) != len(config.market_cap_groups):
        raise ValueError("market_cap_groups 不应包含重复编号")
    if not 0 < config.top_k_pct <= 1:
        raise ValueError("top_k_pct 必须位于 (0, 1] 区间")
    if config.rebalance_days < 1:
        raise ValueError("rebalance_days 必须为正整数")
    if config.lookback_months < 3:
        raise ValueError("lookback_months 至少为 3；研报复现请使用 12")
    if config.half_life_months <= 0:
        raise ValueError("half_life_months 必须大于 0")


def _date_text(value):
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def _progress(stage, completed, total, started_at, current_item=""):
    if total <= 0:
        return
    elapsed = time.time() - started_at
    rate = completed / elapsed if elapsed > 0 else 0.0
    remaining = (total - completed) / rate if rate > 0 else np.nan
    eta = f"{remaining:.0f}s" if np.isfinite(remaining) else "--"
    print(
        f"[{stage}] {completed}/{total} ({completed / total:.1%}) | "
        f"已用 {elapsed:.0f}s | 预计剩余 {eta} | {current_item}"
    )


# =============================================================================
# 2. 日期日程与 BigQuant 数据
# =============================================================================

def build_rebalance_schedule(calendar_dates, start_date, end_date, n_days):
    """返回 (信号日, 成交日)，首次成交使用回测开始日前一交易日信号。"""
    calendar = sorted({_date_text(x) for x in calendar_dates})
    start_text = _date_text(start_date)
    end_text = _date_text(end_date)
    location = {date: i for i, date in enumerate(calendar)}
    trading_dates = [d for d in calendar if start_text <= d <= end_text]

    schedule = []
    for trade_date in trading_dates[::n_days]:
        index = location[trade_date]
        if index > 0:
            schedule.append((calendar[index - 1], trade_date))
    if not schedule:
        raise ValueError("无法生成调仓日程，请检查日期范围")
    return schedule


def get_month_end_dates(calendar_dates):
    dates = pd.Series(pd.to_datetime(sorted(set(calendar_dates))))
    if dates.empty:
        return []
    frame = pd.DataFrame({"date": dates})
    frame["month"] = frame["date"].dt.to_period("M")
    return [
        _date_text(x)
        for x in frame.groupby("month", sort=True)["date"].max().tolist()
    ]


def get_next_trading_date_map(calendar_dates):
    ordered = sorted({_date_text(x) for x in calendar_dates})
    return {
        ordered[index]: ordered[index + 1]
        for index in range(len(ordered) - 1)
    }


def query_calendar(context, history_start):
    sql = """
    SELECT DISTINCT date
    FROM cn_stock_bar1d
    WHERE date >= $history_start AND date <= $end_date
    ORDER BY date
    """
    return dai.query(
        sql,
        params={
            "history_start": _date_text(history_start),
            "end_date": _date_text(context.end_date),
        },
    ).df()["date"].tolist()


def _sql_date_literals(dates):
    """日期来自已查询的交易日历；再次规范化后生成 BigQuant 原生 DATE IN 列表。"""
    normalized = sorted({_date_text(value) for value in dates})
    if not normalized:
        raise ValueError("待查询日期列表不能为空")
    return ", ".join(f"'{date}'" for date in normalized)


def query_required_cross_sections(context, history_start, query_dates):
    """
    使用 BigQuant 文档支持的 date IN ('YYYY-MM-DD', ...) 原生语法。

    不再把日期列表作为 $needed_dates 参数传入：部分 BigQuant 环境会把 DATE 列与
    字符串数组参数比较为空集。下一交易日停牌状态改由额外查询下一交易日截面后在
    pandas 中合并，避免为 LEAD 窗口扫描整个日频历史。
    """
    date_literals = _sql_date_literals(query_dates)
    sql = f"""
    SELECT
        date,
        instrument,
        close,
        open,
        upper_limit,
        lower_limit,
        total_market_cap,
        float_market_cap,
        net_profit_ttm,
        net_profit_deducted_ttm,
        total_owner_equity_lf,
        operating_revenue_ttm,
        sw2014_level1 AS industry,
        st_status,
        suspended,
        list_days
    FROM {DATA_SOURCE}
    WHERE date IN ({date_literals})
    ORDER BY date, instrument
    """
    result = dai.query(
        sql,
        filters={
            "date": [_date_text(history_start), _date_text(context.end_date)]
        },
    ).df()
    if not result.empty:
        return result

    # 单日探针让字段权限、日期覆盖和日期列表匹配问题可以被区分。
    probe_date = sorted({_date_text(x) for x in query_dates})[0]
    probe = dai.query(
        f"""
        SELECT COUNT(*) AS row_count, MIN(date) AS min_date, MAX(date) AS max_date
        FROM {DATA_SOURCE}
        WHERE date = '{probe_date}'
        """,
        filters={"date": [probe_date, probe_date]},
    ).df()
    probe_count = int(probe.iloc[0]["row_count"]) if not probe.empty else -1
    raise ValueError(
        "BigQuant 因子截面查询为空："
        f"查询日期数={len(set(query_dates))}，首个日期={probe_date}，"
        f"单日基础表行数={probe_count}。"
        "若单日行数为0，请检查数据权限/覆盖期；若大于0，请把该完整错误发回。"
    )


def attach_next_day_suspended(raw_data, next_trading_date_map):
    """用额外取得的下一交易日截面生成 next_day_suspended。"""
    base = raw_data.copy()
    base["next_trading_date"] = base["date"].map(next_trading_date_map)
    next_status = raw_data[["date", "instrument", "suspended"]].rename(
        columns={
            "date": "next_trading_date",
            "suspended": "next_day_suspended",
        }
    )
    return base.merge(
        next_status,
        on=["next_trading_date", "instrument"],
        how="left",
        validate="many_to_one",
    )


# =============================================================================
# 3. 原始因子、月度收益标签和历史 RankIC/因子收益
# =============================================================================

def add_raw_factor_values(frame):
    out = frame.copy()
    out["EP"] = out["net_profit_ttm"] / out["total_market_cap"]
    out["EPcut"] = out["net_profit_deducted_ttm"] / out["total_market_cap"]
    out["BP"] = out["total_owner_equity_lf"] / out["total_market_cap"]
    out["SP"] = out["operating_revenue_ttm"] / out["total_market_cap"]
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    return out


def mad_winsorize(series, multiplier=MAD_MULTIPLIER):
    values = pd.to_numeric(series, errors="coerce").astype(float)
    valid = values.dropna()
    if valid.empty:
        return values
    median = valid.median()
    mad = (valid - median).abs().median()
    if not np.isfinite(mad) or mad <= 0:
        return values
    return values.clip(median - multiplier * mad, median + multiplier * mad)


def zscore(series):
    values = pd.to_numeric(series, errors="coerce").astype(float)
    standard_deviation = values.std(ddof=1)
    if not np.isfinite(standard_deviation) or standard_deviation <= 0:
        return pd.Series(np.nan, index=series.index, dtype=float)
    return (values - values.mean()) / standard_deviation


def prepare_exposure_cross_sections(raw_data, exposure_dates):
    """逐截面对四个原始因子做 5MAD 去极值和 Z-score。"""
    needed = set(exposure_dates)
    source = raw_data[raw_data["date"].isin(needed)]
    data_by_date = {d: g for d, g in source.groupby("date", sort=False)}
    pieces = []
    dates = sorted(needed)
    started = time.time()

    for index, date in enumerate(dates, start=1):
        day = data_by_date.get(date)
        if day is None:
            continue
        eligible = day[
            (day["st_status"] == 0)
            & (day["suspended"] == 0)
            & (day["list_days"] >= MIN_LIST_DAYS)
            & day["industry"].notna()
            & day["total_market_cap"].notna()
            & (day["total_market_cap"] > 0)
            & day["float_market_cap"].notna()
            & (day["float_market_cap"] > 0)
        ].drop_duplicates("instrument", keep="last").copy()
        for factor in RAW_FACTORS:
            eligible[factor] = zscore(mad_winsorize(eligible[factor]))
        pieces.append(eligible)

        if index == 1 or index == len(dates) or index % PROGRESS_EVERY == 0:
            _progress("原始因子截面处理", index, len(dates), started, date)

    if not pieces:
        return pd.DataFrame()
    return pd.concat(pieces, ignore_index=True)


def attach_monthly_forward_returns(exposure_panel, raw_data, month_end_dates):
    """以全市场月末交易日到下一月末的收盘收益作为历史估计标签。"""
    ordered_month_ends = sorted(month_end_dates)
    next_month_end = {
        date: ordered_month_ends[index + 1]
        for index, date in enumerate(ordered_month_ends[:-1])
    }
    month_prices = raw_data[raw_data["date"].isin(set(ordered_month_ends))][
        ["date", "instrument", "close"]
    ].drop_duplicates(["date", "instrument"], keep="last")

    labels = month_prices.copy()
    labels["return_end_date"] = labels["date"].map(next_month_end)
    next_prices = month_prices.rename(
        columns={"date": "return_end_date", "close": "next_close"}
    )[["return_end_date", "instrument", "next_close"]]
    labels = labels.merge(
        next_prices,
        on=["return_end_date", "instrument"],
        how="left",
        validate="many_to_one",
    )
    labels["forward_return"] = labels["next_close"] / labels["close"] - 1.0

    monthly = exposure_panel[exposure_panel["date"].isin(set(ordered_month_ends))]
    monthly = monthly.merge(
        labels[["date", "instrument", "return_end_date", "forward_return"]],
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )
    # 与研报复现一致：历史评价剔除截面期下一交易日停牌股票。
    monthly = monthly[monthly["next_day_suspended"] == 0].copy()
    return monthly


def _add_constant(frame):
    out = frame.copy()
    out.insert(0, "const", 1.0)
    return out.astype(float)


def _weighted_regression(y, X, weights):
    X_values = X.to_numpy(dtype=float)
    y_values = y.to_numpy(dtype=float)
    root_weight = np.sqrt(weights.to_numpy(dtype=float))
    X_weighted = X_values * root_weight[:, None]
    y_weighted = y_values * root_weight
    beta, _, rank, _ = np.linalg.lstsq(X_weighted, y_weighted, rcond=None)
    if len(y_values) - int(rank) <= 0:
        raise np.linalg.LinAlgError("WLS 自由度不足")
    return pd.Series(beta, index=X.columns)


def cross_section_factor_metrics(day, factor):
    columns = [
        factor,
        "forward_return",
        "total_market_cap",
        "float_market_cap",
        "industry",
    ]
    sample = day[columns].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(sample) < MIN_CROSS_SECTION or sample[factor].nunique() < 3:
        return np.nan, np.nan, len(sample)

    factor_return = np.nan
    rank_ic = np.nan
    try:
        industry = pd.get_dummies(
            sample["industry"].astype(str), prefix="ind", drop_first=True, dtype=float
        )
        controls = pd.DataFrame(
            {
                factor: sample[factor].astype(float),
                "ln_total_market_cap": np.log(sample["total_market_cap"].astype(float)),
            },
            index=sample.index,
        )
        design = _add_constant(pd.concat([controls, industry], axis=1))
        params = _weighted_regression(
            sample["forward_return"].astype(float),
            design,
            np.sqrt(sample["float_market_cap"].astype(float)),
        )
        factor_return = float(params[factor])
    except Exception:
        factor_return = np.nan

    try:
        neutral_design = _add_constant(
            pd.concat(
                [
                    pd.Series(
                        np.log(sample["total_market_cap"].astype(float)),
                        index=sample.index,
                        name="ln_total_market_cap",
                    ),
                    industry,
                ],
                axis=1,
            )
        )
        beta = np.linalg.lstsq(
            neutral_design.to_numpy(dtype=float),
            sample[factor].to_numpy(dtype=float),
            rcond=None,
        )[0]
        residual = pd.Series(
            sample[factor].to_numpy(dtype=float)
            - neutral_design.to_numpy(dtype=float) @ beta,
            index=sample.index,
        )
        rank_ic = float(residual.corr(sample["forward_return"], method="spearman"))
    except Exception:
        rank_ic = np.nan

    return factor_return, rank_ic, len(sample)


def build_raw_metric_history(monthly_panel):
    records = []
    dates = sorted(monthly_panel.loc[monthly_panel["forward_return"].notna(), "date"].unique())
    total = len(dates) * len(RAW_FACTORS)
    completed = 0
    started = time.time()

    for date in dates:
        day = monthly_panel[monthly_panel["date"] == date]
        end_dates = day["return_end_date"].dropna().unique()
        return_end_date = _date_text(end_dates[0]) if len(end_dates) else None
        for factor in RAW_FACTORS:
            factor_return, rank_ic, sample_count = cross_section_factor_metrics(day, factor)
            records.append(
                {
                    "date": _date_text(date),
                    "return_end_date": return_end_date,
                    "factor": factor,
                    "factor_return": factor_return,
                    "rank_ic": rank_ic,
                    "sample_count": sample_count,
                }
            )
            completed += 1
        if completed == total or completed % max(PROGRESS_EVERY, len(RAW_FACTORS)) == 0:
            _progress("历史月度收益与RankIC", completed, total, started, _date_text(date))

    return pd.DataFrame(records)


# =============================================================================
# 4. 九种研报合成方法
# =============================================================================

def normalize_signed(raw):
    values = np.asarray(raw, dtype=float)
    denominator = values.sum()
    if not np.all(np.isfinite(values)) or abs(denominator) < 1e-12:
        return np.repeat(1.0 / len(values), len(values))
    return values / denominator


def exponential_time_weights(length, half_life):
    distance = np.arange(length - 1, -1, -1, dtype=float)
    weights = np.power(0.5, distance / half_life)
    return weights / weights.sum()


def solve_nonnegative_ratio(mu, covariance):
    """max w'mu/sqrt(w'Cov w)，约束 w>=0、sum(w)=1。"""
    mu = np.asarray(mu, dtype=float)
    covariance = np.asarray(covariance, dtype=float)
    factor_count = len(mu)
    covariance = (covariance + covariance.T) / 2.0
    covariance = covariance + np.eye(factor_count) * 1e-10

    def objective(weight):
        variance = float(weight @ covariance @ weight)
        if variance <= 1e-16:
            return 1e6
        return -float(weight @ mu) / np.sqrt(variance)

    result = minimize(
        objective,
        x0=np.repeat(1.0 / factor_count, factor_count),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * factor_count,
        constraints=[{"type": "eq", "fun": lambda w: float(w.sum() - 1.0)}],
        options={"maxiter": 500, "ftol": 1e-12, "disp": False},
    )
    if not result.success or not np.all(np.isfinite(result.x)):
        positive = np.maximum(mu, 0.0)
        if positive.sum() > 0:
            return positive / positive.sum()
        return np.repeat(1.0 / factor_count, factor_count)
    weights = np.maximum(result.x, 0.0)
    return weights / weights.sum()


def compute_composite_weight(method, X, historical_ic, historical_return, half_life):
    equal_weight = np.repeat(1.0 / len(RAW_FACTORS), len(RAW_FACTORS))

    if method == "fac_eqwt":
        return equal_weight
    if method == "fac_pca1":
        loading = PCA(n_components=1).fit(X.to_numpy(dtype=float)).components_[0]
        loading = loading.astype(float)
        if loading.sum() < 0:
            loading *= -1.0
        return loading

    mu_ic = historical_ic.mean(axis=0).to_numpy(dtype=float)
    mu_return = historical_return.mean(axis=0).to_numpy(dtype=float)
    decay = exponential_time_weights(len(historical_ic), half_life)

    if method == "fac_ret":
        return normalize_signed(mu_return)
    if method == "fac_ret_half":
        return normalize_signed(
            (historical_return.to_numpy(dtype=float) * decay[:, None]).sum(axis=0)
        )
    if method == "fac_ic":
        return normalize_signed(mu_ic)
    if method == "fac_ic_half":
        return normalize_signed(
            (historical_ic.to_numpy(dtype=float) * decay[:, None]).sum(axis=0)
        )
    if method == "fac_maxicir_samp":
        return solve_nonnegative_ratio(
            mu_ic,
            historical_ic.cov(ddof=1).to_numpy(dtype=float),
        )
    if method == "fac_maxicir1":
        covariance = LedoitWolf().fit(
            historical_ic.to_numpy(dtype=float)
        ).covariance_
        return solve_nonnegative_ratio(mu_ic, covariance)
    if method == "fac_maxic":
        value_covariance = LedoitWolf().fit(X.to_numpy(dtype=float)).covariance_
        return solve_nonnegative_ratio(mu_ic, value_covariance)
    raise ValueError(f"未知合成方法：{method}")


def build_test_factor_exposures(exposure_panel, signal_dates, metric_history, config):
    """为每个信号日生成指定研报因子；历史标签必须已经实现完毕。"""
    signal_set = set(signal_dates)
    signal_source = exposure_panel[exposure_panel["date"].isin(signal_set)]
    data_by_date = {d: g for d, g in signal_source.groupby("date", sort=False)}
    output = []
    weight_records = []

    if config.factor_name in HISTORY_WEIGHTED_FACTORS:
        ic_history = metric_history.pivot(index="date", columns="factor", values="rank_ic")
        return_history = metric_history.pivot(
            index="date", columns="factor", values="factor_return"
        )
        ic_history = ic_history.reindex(columns=RAW_FACTORS).sort_index()
        return_history = return_history.reindex(columns=RAW_FACTORS).sort_index()

    started = time.time()
    dates = sorted(signal_set)
    for index, signal_date in enumerate(dates, start=1):
        current = data_by_date.get(signal_date)
        if current is None or current.empty:
            continue
        current = current.copy()

        if config.factor_name in RAW_FACTORS:
            current["factor_value"] = current[config.factor_name]
        else:
            X = current[RAW_FACTORS].fillna(0.0).astype(float)
            if len(X) < MIN_CROSS_SECTION:
                continue

            if config.factor_name in HISTORY_WEIGHTED_FACTORS:
                completed_metrics = metric_history[
                    metric_history["return_end_date"].notna()
                    & (metric_history["return_end_date"] <= signal_date)
                ]
                complete_dates = sorted(completed_metrics["date"].unique())
                complete_dates = complete_dates[-config.lookback_months :]
                historical_ic = ic_history.reindex(complete_dates).dropna(how="any")
                historical_return = return_history.reindex(complete_dates).dropna(how="any")
                shared_dates = historical_ic.index.intersection(historical_return.index)
                historical_ic = historical_ic.loc[shared_dates].tail(config.lookback_months)
                historical_return = historical_return.loc[shared_dates].tail(
                    config.lookback_months
                )

                if len(shared_dates) < config.lookback_months:
                    print(
                        f"[警告] {signal_date} 只有 {len(shared_dates)} 个完整历史月，"
                        f"不足 {config.lookback_months}，跳过该次信号"
                    )
                    continue
                history_months = len(shared_dates)
                history_start = min(shared_dates)
                history_end = max(shared_dates)
                latest_metric_end = completed_metrics[
                    completed_metrics["date"].isin(shared_dates)
                ]["return_end_date"].max()
            else:
                historical_ic = pd.DataFrame(columns=RAW_FACTORS)
                historical_return = pd.DataFrame(columns=RAW_FACTORS)
                history_months = 0
                history_start = None
                history_end = None
                latest_metric_end = None

            weight = compute_composite_weight(
                config.factor_name,
                X,
                historical_ic,
                historical_return,
                config.half_life_months,
            )
            current["factor_value"] = zscore(
                pd.Series(X.to_numpy(dtype=float) @ weight, index=current.index)
            )
            for raw_factor, value in zip(RAW_FACTORS, weight):
                weight_records.append(
                    {
                        "signal_date": signal_date,
                        "method": config.factor_name,
                        "raw_factor": raw_factor,
                        "weight": float(value),
                        "history_months": history_months,
                        "history_start": history_start,
                        "history_end": history_end,
                        "latest_realized_return_end": latest_metric_end,
                    }
                )

        output.append(
            current[
                [
                    "date",
                    "instrument",
                    "total_market_cap",
                    "factor_value",
                ]
            ]
        )
        if index == 1 or index == len(dates) or index % PROGRESS_EVERY == 0:
            _progress("目标因子生成", index, len(dates), started, signal_date)

    factor_panel = pd.concat(output, ignore_index=True) if output else pd.DataFrame()
    return factor_panel, pd.DataFrame(weight_records)


# =============================================================================
# 5. 市值分组与组内选股
# =============================================================================

def assign_market_cap_groups(cross_section, group_count=MARKET_CAP_GROUP_COUNT):
    frame = cross_section.sort_values(
        ["total_market_cap", "instrument"], ascending=[True, True]
    ).copy()
    if len(frame) < group_count:
        return frame.iloc[0:0].assign(market_cap_group=pd.Series(dtype="int64"))
    cap_rank = frame["total_market_cap"].rank(method="first", ascending=True)
    frame["market_cap_group"] = pd.qcut(
        cap_rank,
        q=group_count,
        labels=list(range(1, group_count + 1)),
    ).astype(int)
    return frame


def select_one_signal_date(day_data, config):
    universe = day_data[
        day_data["total_market_cap"].notna()
        & (day_data["total_market_cap"] > 0)
    ].drop_duplicates("instrument", keep="last")
    grouped = assign_market_cap_groups(universe)
    if grouped.empty:
        return [], []

    selected = []
    audit = []
    positive = config.factor_direction == 1
    for group_number in config.market_cap_groups:
        group_all = grouped[grouped["market_cap_group"] == group_number]
        group_valid = group_all[
            group_all["factor_value"].notna()
            & np.isfinite(group_all["factor_value"].astype(float))
        ]
        select_count = (
            max(1, int(math.ceil(len(group_valid) * config.top_k_pct)))
            if len(group_valid)
            else 0
        )
        picked = group_valid.sort_values(
            ["factor_value", "instrument"],
            ascending=[not positive, True],
        ).head(select_count)
        selected.extend(picked["instrument"].tolist())
        audit.append(
            {
                "market_cap_group": group_number,
                "group_stock_count": len(group_all),
                "valid_factor_count": len(group_valid),
                "selected_count": len(picked),
            }
        )
    return list(dict.fromkeys(selected)), audit


def build_targets_and_audit(factor_panel, schedule, config):
    data_by_date = {d: g for d, g in factor_panel.groupby("date", sort=False)}
    targets_by_trade_date = {}
    audit_records = []
    started = time.time()

    for index, (signal_date, trade_date) in enumerate(schedule, start=1):
        day = data_by_date.get(signal_date)
        if day is None or day.empty:
            continue
        targets, day_audit = select_one_signal_date(day, config)
        if not targets:
            continue
        targets_by_trade_date[trade_date] = targets
        for row in day_audit:
            row.update(
                {
                    "signal_date": signal_date,
                    "trade_date": trade_date,
                    "factor_name": config.factor_name,
                    "target_stock_count": len(targets),
                }
            )
            audit_records.append(row)
        if index == 1 or index == len(schedule) or index % PROGRESS_EVERY == 0:
            _progress("市值分组与选股", index, len(schedule), started, signal_date)

    return targets_by_trade_date, pd.DataFrame(audit_records)


def build_trade_status(raw_data, trade_dates):
    columns = ["open", "upper_limit", "lower_limit", "st_status", "suspended"]
    result = {}
    for date, group in raw_data[raw_data["date"].isin(set(trade_dates))].groupby(
        "date", sort=False
    ):
        unique = group.drop_duplicates("instrument", keep="last").set_index("instrument")
        result[date] = unique[columns].to_dict("index")
    return result


# =============================================================================
# 6. BigTrader 初始化与成交约束
# =============================================================================

def initialize(context):
    config = _ACTIVE_CONFIG
    if config is None:
        raise RuntimeError("请通过 run_backtest(...) 启动策略")
    validate_config(config)
    context.config = config
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COMMISSION,
        )
    )

    # 12 个可用月度标签至少需要 13 个历史月末，再多取 2 个月作为安全缓冲。
    history_start = (
        pd.Timestamp(context.start_date)
        - pd.DateOffset(months=config.lookback_months + 3)
    ).replace(day=1)

    print("[1/7] 获取交易日历、月末截面和调仓日程")
    calendar_dates = query_calendar(context, history_start)
    next_trading_date_map = get_next_trading_date_map(calendar_dates)
    schedule = build_rebalance_schedule(
        calendar_dates,
        context.start_date,
        context.end_date,
        config.rebalance_days,
    )
    month_end_dates = get_month_end_dates(calendar_dates)
    signal_dates = [signal for signal, _ in schedule]
    trade_dates = [trade for _, trade in schedule]
    needs_monthly_history = config.factor_name in HISTORY_WEIGHTED_FACTORS
    history_dates = month_end_dates if needs_monthly_history else []
    next_status_dates = [
        next_trading_date_map[date]
        for date in history_dates
        if date in next_trading_date_map
    ]
    query_dates = sorted(
        set(history_dates + signal_dates + trade_dates + next_status_dates)
    )

    print(
        f"[2/7] 查询所需截面 | 因子={config.factor_name} "
        f"({FACTOR_MEANINGS[config.factor_name]}) | T={config.lookback_months}月"
    )
    raw_data = query_required_cross_sections(
        context,
        history_start,
        query_dates,
    )
    raw_data["date"] = pd.to_datetime(raw_data["date"]).dt.strftime("%Y-%m-%d")
    raw_data["instrument"] = raw_data["instrument"].astype(str)
    raw_data = attach_next_day_suspended(raw_data, next_trading_date_map)
    raw_data = add_raw_factor_values(raw_data)
    print(
        f"[数据] {len(raw_data):,} 行 | {raw_data['instrument'].nunique():,} 只股票 | "
        f"{raw_data['date'].min()} 至 {raw_data['date'].max()}"
    )

    print("[3/7] 对月末和信号日原始因子做5MAD去极值与截面标准化")
    exposure_dates = sorted(set(history_dates + signal_dates))
    exposure_panel = prepare_exposure_cross_sections(raw_data, exposure_dates)
    if exposure_panel.empty:
        raise ValueError("原始因子截面处理后没有有效数据")

    if needs_monthly_history:
        print("[4/7] 构造月末至下一月末收益，并估计原始因子历史收益与RankIC")
        monthly_panel = attach_monthly_forward_returns(
            exposure_panel,
            raw_data,
            month_end_dates,
        )
        metric_history = build_raw_metric_history(monthly_panel)
    else:
        print(f"[4/7] {config.factor_name} 不需要历史权重，跳过月度收益与RankIC估计")
        metric_history = pd.DataFrame(
            columns=[
                "date",
                "return_end_date",
                "factor",
                "factor_return",
                "rank_ic",
                "sample_count",
            ]
        )

    print(f"[5/7] 生成目标因子 {config.factor_name}")
    factor_panel, weight_history = build_test_factor_exposures(
        exposure_panel,
        signal_dates,
        metric_history,
        config,
    )
    if factor_panel.empty:
        raise ValueError("目标因子没有生成有效信号，请检查历史预热期和数据字段")

    print("[6/7] 完成15组市值分位，并在指定组内独立选择Top/Bottom k%")
    context.targets_by_trade_date, selection_audit = build_targets_and_audit(
        factor_panel,
        schedule,
        config,
    )
    context.trade_status_by_date = build_trade_status(raw_data, trade_dates)
    if not context.targets_by_trade_date:
        raise ValueError("所有调仓日均无有效候选股票")

    RESEARCH_ARTIFACTS.clear()
    RESEARCH_ARTIFACTS.update(
        {
            "config": config,
            "raw_metric_history": metric_history,
            "weight_history": weight_history,
            "selection_audit": selection_audit,
        }
    )
    context.selection_audit = selection_audit
    context.weight_history = weight_history
    context.blocked_order_count = 0
    average_count = np.mean(
        [len(stocks) for stocks in context.targets_by_trade_date.values()]
    )
    print(
        f"[7/7] 初始化完成 | 有效调仓={len(context.targets_by_trade_date)}次 | "
        f"平均目标={average_count:.1f}只 | 市值组={list(config.market_cap_groups)} | "
        f"t日收盘信号、t+1日开盘成交"
    )


def _current_weight(context, instrument):
    position = context.portfolio.positions.get(instrument)
    if position is None:
        return 0.0
    market_value = getattr(position, "market_value", 0.0) or 0.0
    portfolio_value = getattr(context.portfolio, "portfolio_value", 0.0) or 0.0
    return float(market_value) / float(portfolio_value) if portfolio_value > 0 else 0.0


def _finite_number(value):
    try:
        return np.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def _status_int(value, default):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except (TypeError, ValueError):
        return default


def _can_adjust_to_weight(context, instrument, target_weight, status):
    if status is None or _status_int(status.get("suspended"), 1) != 0:
        return False, "停牌或缺少状态"
    open_price = status.get("open")
    if not _finite_number(open_price) or float(open_price) <= 0:
        return False, "无有效开盘价"

    current_weight = _current_weight(context, instrument)
    increase = target_weight > current_weight + MIN_WEIGHT_CHANGE
    decrease = target_weight < current_weight - MIN_WEIGHT_CHANGE
    if not increase and not decrease:
        return False, "无需调整"
    if increase and _status_int(status.get("st_status"), 1) != 0:
        return False, "ST股票不买入"

    upper_limit = status.get("upper_limit")
    lower_limit = status.get("lower_limit")
    if increase and _finite_number(upper_limit):
        if float(open_price) >= float(upper_limit) * (1 - 1e-7):
            return False, "开盘涨停不可买"
    if decrease and _finite_number(lower_limit):
        if float(open_price) <= float(lower_limit) * (1 + 1e-7):
            return False, "开盘跌停不可卖"
    return True, "可交易"


def handle_data(context, data):
    today = data.current_dt.strftime("%Y-%m-%d")
    targets = context.targets_by_trade_date.get(today)
    if targets is None:
        return

    status_today = context.trade_status_by_date.get(today, {})
    target_set = set(targets)
    target_weight = 1.0 / len(targets)
    current_set = set()
    for instrument, position in context.portfolio.positions.items():
        amount = getattr(position, "amount", None)
        if amount is None:
            amount = getattr(position, "current_qty", 0)
        if (amount or 0) > 0:
            current_set.add(instrument)

    submitted = 0
    blocked = 0
    for instrument in sorted(current_set - target_set):
        allowed, _ = _can_adjust_to_weight(
            context, instrument, 0.0, status_today.get(instrument)
        )
        if allowed:
            context.order_target_percent(instrument, 0.0)
            submitted += 1
        else:
            blocked += 1

    for instrument in sorted(targets):
        allowed, reason = _can_adjust_to_weight(
            context, instrument, target_weight, status_today.get(instrument)
        )
        if allowed:
            context.order_target_percent(instrument, target_weight)
            submitted += 1
        elif reason != "无需调整":
            blocked += 1

    context.blocked_order_count += blocked
    if LOG_EACH_REBALANCE or blocked > 0:
        context.logger.info(
            f"{today} 调仓 | 因子={context.config.factor_name} | "
            f"市值组={list(context.config.market_cap_groups)} | "
            f"目标={len(targets)} | 提交={submitted} | 受限={blocked}"
        )


# =============================================================================
# 7. 参数化运行接口
# =============================================================================

def run_backtest(
    factor_name=FACTOR_TO_TEST,
    factor_direction=FACTOR_DIRECTION,
    market_cap_groups=None,
    top_k_pct=TOP_K_PCT,
    rebalance_days=REBALANCE_DAYS,
    lookback_months=LOOKBACK_MONTHS,
    half_life_months=HALF_LIFE_MONTHS,
):
    """
    例：
        run_backtest("EP", 1, [1, 2, 3], 0.10, 20)
        run_backtest("fac_maxicir1", 1, [13, 14, 15], 0.20, 10)
    """
    global _ACTIVE_CONFIG
    selected_groups = MARKET_CAP_GROUPS if market_cap_groups is None else market_cap_groups
    _ACTIVE_CONFIG = make_config(
        factor_name=factor_name,
        factor_direction=factor_direction,
        market_cap_groups=selected_groups,
        top_k_pct=top_k_pct,
        rebalance_days=rebalance_days,
        lookback_months=lookback_months,
        half_life_months=half_life_months,
    )
    return bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date=START_DATE,
        end_date=END_DATE,
        capital_base=CAPITAL_BASE,
        benchmark=BENCHMARK,
        initialize=initialize,
        handle_data=handle_data,
        order_price_field_buy="open",
        order_price_field_sell="open",
        volume_limit=VOLUME_LIMIT,
    )


if __name__ == "__main__":
    performance = run_backtest(
        factor_name=FACTOR_TO_TEST,
        factor_direction=FACTOR_DIRECTION,
        market_cap_groups=MARKET_CAP_GROUPS,
        top_k_pct=TOP_K_PCT,
        rebalance_days=REBALANCE_DAYS,
        lookback_months=LOOKBACK_MONTHS,
        half_life_months=HALF_LIFE_MONTHS,
    )


[2026-07-16 16:35:54] [info     ] bigtrader init ..
[2026-07-16 16:35:54] [info     ] bigtrader.run start: market=cn_stock, frequency=1d, mode=backtest, account_type=STOCK, start_date=2022-01-01, end_date=2026-06-30


[2026-07-16 16:35:57] [info     ] bigtrader<backtest> init ..
[2026-07-16 16:35:57] [info     ] prepare data ..
[2026-07-16 16:35:57] [info     ] bar1d_df: (5890930, 16)
[2026-07-16 16:36:13] [info     ] bigtrader use dividend data: (19841, 8)
[1/7] 获取交易日历、月末截面和调仓日程
[2/7] 查询所需截面 | 因子=fac_maxicir1 (最大化 IC_IR 复合因子（压缩协方差）) | T=12月
[数据] 883,263 行 | 5,983 只股票 | 2020-10-30 至 2026-06-30
[3/7] 对月末和信号日原始因子做5MAD去极值与截面标准化
[原始因子截面处理] 1/86 (1.2%) | 已用 0s | 预计剩余 1s | 2020-10-30
[原始因子截面处理] 20/86 (23.3%) | 已用 0s | 预计剩余 0s | 2022-04-29
[原始因子截面处理] 40/86 (46.5%) | 已用 0s | 预计剩余 0s | 2023-08-31
[原始因子截面处理] 60/86 (69.8%) | 已用 0s | 预计剩余 0s | 2024-11-29
[原始因子截面处理] 80/86 (93.0%) | 已用 1s | 预计剩余 0s | 2026-02-27
[原始因子截面处理] 86/86 (100.0%) | 已用 1s | 预计剩余 0s | 2026-06-30
[4/7] 构造月末至下一月末收益，并估计原始因子历史收益与RankIC
[历史月度收益与RankIC] 20/272 (7.4%) | 已用 3s | 预计剩余 32s | 2021-02-26
[历史月度收益与RankIC] 40/272 (14.7%) | 已用 6s | 预计剩余 33s | 2021-07-30
[历史月度收益与RankIC] 60/272 (22.1%) | 已用 10s | 预计剩余 34s | 2021-12-31
[历史月度收益与RankIC] 80/272 (

[2026-07-16 16:37:34] [info     ] bigtrader run done.


: 